# Getting Started with Dynamic Tables - Healthcare Edition

This notebook demonstrates Snowflake Dynamic Tables using a healthcare dataset with patients, appointments, and doctors. Dynamic Tables provide a reliable, cost-effective, and automated way to transform healthcare data using a declarative approach, automatically refreshing when underlying data changes.

**Original Quickstart:** https://quickstarts.snowflake.com/guide/getting_started_with_dynamic_tables/index.html

**Healthcare Use Cases:**
- Real-time patient appointment tracking
- Doctor utilization and scheduling analytics
- Patient care metrics and reporting
- Automated healthcare dashboard updates

**What you'll learn:**
- How to create and configure Dynamic Tables for healthcare data
- Understanding TARGET_LAG and refresh behavior for medical workflows
- Working with incremental processing for patient data
- Monitoring Dynamic Table performance in healthcare scenarios

**Prerequisites:** 
- Snowflake account with ACCOUNTADMIN or equivalent privileges
- Access to a warehouse for compute resources


In [ ]:
-- Check session context and current role
SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA();


In [ ]:
-- IMPORTANT!!! - Set your name HERE
SET my_name = 'Ginu';

-- Setup: Create database and schema for Healthcare Dynamic Tables demo
SET schema_name = UPPER('patient_care_' || $my_name);
USE DATABASE SNOWDAY_DB;
CREATE OR REPLACE SCHEMA IDENTIFIER($schema_name);


![Alt text](https://github.com/sfc-gh-gkuncheria/dynamic-tables-hol/blob/main/images/dt_start.png?raw=true)




In [ ]:
-- Create base tables for healthcare data
CREATE OR REPLACE TABLE patients (
    patient_id NUMBER,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    date_of_birth DATE,
    gender VARCHAR(10),
    phone VARCHAR(15),
    email VARCHAR(100),
    insurance_provider VARCHAR(50),
    registration_date DATE
);

CREATE OR REPLACE TABLE doctors (
    doctor_id NUMBER,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    specialty VARCHAR(100),
    department VARCHAR(50),
    phone VARCHAR(15),
    email VARCHAR(100),
    hire_date DATE
);

CREATE OR REPLACE TABLE appointments (
    appointment_id NUMBER,
    patient_id NUMBER,
    doctor_id NUMBER,
    appointment_date DATE,
    appointment_time TIME,
    duration_minutes NUMBER,
    appointment_type VARCHAR(50),
    status VARCHAR(20),
    created_timestamp TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);


In [ ]:
-- Insert sample data into patients table
INSERT INTO patients VALUES
(1001, 'John', 'Smith', '1985-03-15', 'Male', '555-0101', 'john.smith@email.com', 'Blue Cross', '2023-01-15'),
(1002, 'Sarah', 'Johnson', '1992-07-22', 'Female', '555-0102', 'sarah.johnson@email.com', 'Aetna', '2023-02-20'),
(1003, 'Michael', 'Brown', '1978-11-08', 'Male', '555-0103', 'michael.brown@email.com', 'United Healthcare', '2023-03-10'),
(1004, 'Emily', 'Davis', '1990-05-12', 'Female', '555-0104', 'emily.davis@email.com', 'Cigna', '2023-04-05'),
(1005, 'Robert', 'Wilson', '1965-09-30', 'Male', '555-0105', 'robert.wilson@email.com', 'Medicare', '2023-05-12');

In [ ]:
-- Insert sample data into doctors table
INSERT INTO doctors VALUES
(2001, 'Dr. Maria', 'Garcia', 'Cardiology', 'Internal Medicine', '555-1001', 'maria.garcia@hospital.com', '2020-01-15'),
(2002, 'Dr. James', 'Chen', 'Pediatrics', 'Family Medicine', '555-1002', 'james.chen@hospital.com', '2019-03-20'),
(2003, 'Dr. Lisa', 'Thompson', 'Orthopedics', 'Surgery', '555-1003', 'lisa.thompson@hospital.com', '2021-06-10'),
(2004, 'Dr. David', 'Rodriguez', 'Dermatology', 'Specialty Care', '555-1004', 'david.rodriguez@hospital.com', '2018-09-05'),
(2005, 'Dr. Jennifer', 'Lee', 'General Practice', 'Family Medicine', '555-1005', 'jennifer.lee@hospital.com', '2022-02-12');

In [ ]:
-- Insert initial appointments data
INSERT INTO appointments (appointment_id, patient_id, doctor_id, appointment_date, appointment_time, duration_minutes, appointment_type, status) VALUES
(3001, 1001, 2001, '2024-01-10', '09:00:00', 30, 'Consultation', 'Completed'),
(3002, 1002, 2002, '2024-01-11', '10:30:00', 45, 'Check-up', 'Completed'),
(3003, 1003, 2003, '2024-01-12', '14:00:00', 60, 'Surgery Consultation', 'Completed'),
(3004, 1004, 2004, '2024-01-13', '11:15:00', 30, 'Follow-up', 'Completed'),
(3005, 1005, 2005, '2024-01-14', '15:30:00', 30, 'Annual Physical', 'Completed');


## Creating Dynamic Tables

Now we'll create Dynamic Tables that automatically maintain aggregated views of our healthcare data. Dynamic Tables use the `TARGET_LAG` parameter to control refresh frequency - critical for real-time patient care monitoring.


In [ ]:
-- Create Dynamic Table: Appointment Details with Patient and Doctor Info
CREATE OR REPLACE DYNAMIC TABLE appointment_details
TARGET_LAG = '1 minute'
WAREHOUSE = COMPUTE_WH
AS
SELECT 
    a.appointment_id,
    a.appointment_date,
    a.appointment_time,
    CONCAT(p.first_name, ' ', p.last_name) AS patient_name,
    p.insurance_provider,
    DATEDIFF('year', p.date_of_birth, CURRENT_DATE()) AS patient_age,
    CONCAT(d.first_name, ' ', d.last_name) AS doctor_name,
    d.specialty,
    d.department,
    a.appointment_type,
    a.duration_minutes,
    a.status
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id  
JOIN doctors d ON a.doctor_id = d.doctor_id;


In [ ]:
-- Create Dynamic Table: Daily Appointment Summary
CREATE OR REPLACE DYNAMIC TABLE daily_appointment_summary  
TARGET_LAG = '5 minutes'
WAREHOUSE = COMPUTE_WH
AS
SELECT 
    appointment_date,
    department,
    specialty,
    COUNT(*) AS total_appointments,
    COUNT(DISTINCT patient_name) AS unique_patients,
    SUM(duration_minutes) AS total_appointment_time,
    AVG(duration_minutes) AS avg_appointment_duration,
    COUNT(CASE WHEN status = 'Completed' THEN 1 END) AS completed_appointments,
    COUNT(CASE WHEN status = 'Scheduled' THEN 1 END) AS scheduled_appointments
FROM appointment_details
GROUP BY appointment_date, department, specialty;


In [ ]:
-- Query the initial state of our Dynamic Tables
SELECT * FROM appointment_details ORDER BY appointment_id;


In [ ]:
-- View the daily appointment summary
SELECT * FROM daily_appointment_summary ORDER BY appointment_date, department, specialty;


## Testing Dynamic Table Updates

Now let's add new healthcare data to our base tables and observe how the Dynamic Tables automatically update to reflect new patients, doctors, and appointments.


In [ ]:
-- Add new patients to expand our patient base
INSERT INTO patients VALUES
(1006, 'Anna', 'Martinez', '1988-12-03', 'Female', '555-0106', 'anna.martinez@email.com', 'Humana', '2024-01-15'),
(1007, 'William', 'Taylor', '1995-04-18', 'Male', '555-0107', 'william.taylor@email.com', 'Kaiser', '2024-01-16'),
(1008, 'Jessica', 'Anderson', '1982-08-25', 'Female', '555-0108', 'jessica.anderson@email.com', 'Blue Cross', '2024-01-17');


In [ ]:
-- Add new doctors to expand medical staff
INSERT INTO doctors VALUES
(2006, 'Dr. Kevin', 'Park', 'Neurology', 'Specialty Care', '555-1006', 'kevin.park@hospital.com', '2023-08-15'),
(2007, 'Dr. Rachel', 'White', 'Emergency Medicine', 'Emergency Department', '555-1007', 'rachel.white@hospital.com', '2023-11-20');


In [ ]:
-- Add new appointments that will trigger Dynamic Table updates
INSERT INTO appointments (appointment_id, patient_id, doctor_id, appointment_date, appointment_time, duration_minutes, appointment_type, status) VALUES
(3006, 1006, 2006, '2024-01-15', '09:30:00', 45, 'Neurological Exam', 'Scheduled'),
(3007, 1007, 2007, '2024-01-15', '14:00:00', 30, 'Emergency Visit', 'Completed'), 
(3008, 1008, 2001, '2024-01-16', '10:00:00', 30, 'Cardiac Follow-up', 'Scheduled'),
(3009, 1001, 2005, '2024-01-16', '16:30:00', 30, 'Annual Check-up', 'Scheduled'),
(3010, 1002, 2002, '2024-01-17', '11:00:00', 45, 'Pediatric Consultation', 'Scheduled');


In [ ]:
-- Check updated appointment details (Dynamic Table should include new appointments)
SELECT * FROM appointment_details WHERE appointment_id >= 3006 ORDER BY appointment_id;


In [ ]:
-- View updated daily healthcare summary with new data
SELECT * FROM daily_appointment_summary ORDER BY appointment_date, department, specialty;


## Monitoring Dynamic Tables

Let's examine the metadata and performance of our Dynamic Tables.


In [ ]:
-- View Dynamic Table information and status
SELECT * 
FROM 
    TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE 
    SCHEMA_NAME IN ($schema_name)
    -- AND REFRESH_ACTION != 'NO_DATA'
ORDER BY 
    DATA_TIMESTAMP DESC, REFRESH_END_TIME DESC LIMIT 10;


## You can use Snowsight GUI to visualize and monitor the directed acyclic graph (DAG) of your pipeline
![Alt text](https://github.com/sfc-gh-gkuncheria/dynamic-tables-hol/blob/main/images/dt_dag.png?raw=trueg)


## So far we have built the following:
![Alt text](https://github.com/sfc-gh-gkuncheria/dynamic-tables-hol/blob/main/images/dt_refresh_time.png?raw=true)


## Advanced Dynamic Table Features

Let's explore some advanced capabilities including manual refresh and downstream dependencies.

![Alt text](https://github.com/sfc-gh-gkuncheria/dynamic-tables-hol/blob/main/images/dt_downstream.png?raw=true)


In [ ]:
SHOW DYNAMIC TABLES;

In [ ]:
ALTER DYNAMIC TABLE APPOINTMENT_DETAILS SET TARGET_LAG = DOWNSTREAM;
ALTER DYNAMIC TABLE daily_appointment_summary SET TARGET_LAG = DOWNSTREAM;

In [ ]:
SHOW DYNAMIC TABLES;

In [ ]:
-- Create a Dynamic Table that depends on another Dynamic Table
CREATE OR REPLACE DYNAMIC TABLE monthly_appointments_rollup
TARGET_LAG = '1 hour'  
WAREHOUSE = COMPUTE_WH
AS
SELECT 
    DATE_TRUNC('month', appointment_date) AS care_month,
    department,
    SUM(total_appointments) AS monthly_appointments,
    SUM(unique_patients) AS monthly_unique_patients,
    SUM(total_appointment_time) AS monthly_appointment_time,
    AVG(avg_appointment_duration) AS avg_monthly_duration,
    SUM(completed_appointments) AS monthly_completed,
    SUM(scheduled_appointments) AS monthly_scheduled
FROM daily_appointment_summary
GROUP BY DATE_TRUNC('month', appointment_date), department;


In [ ]:
-- View the monthly healthcare rollup data
SELECT * FROM monthly_appointments_rollup ORDER BY care_month, department;


In [ ]:
-- FORCE A REFRESH
ALTER DYNAMIC TABLE monthly_appointments_rollup REFRESH;

-- Check REFRESH_START_TIME
SELECT REFRESH_START_TIME, * 
FROM 
    TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE 
    SCHEMA_NAME IN ($schema_name)
    -- AND REFRESH_ACTION != 'NO_DATA'
ORDER BY 
    DATA_TIMESTAMP DESC, REFRESH_END_TIME DESC LIMIT 10;

## Key Takeaways

**What we learned about Dynamic Tables in Healthcare:**

1. **Declarative Approach**: Define the end state, Snowflake handles the healthcare data pipeline
2. **Automatic Refresh**: Tables update based on TARGET_LAG when patient/appointment data changes  
3. **Incremental Processing**: Only processes changed healthcare data for efficiency
4. **Dependency Management**: Downstream healthcare reports automatically refresh when upstream data updates
5. **Monitoring**: Rich metadata available through INFORMATION_SCHEMA views for healthcare analytics

**Healthcare Use Cases for Dynamic Tables:**
- **Real-time Patient Monitoring**: 1-minute refresh for critical care dashboards
- **Doctor Utilization Reports**: 5-minute refresh for scheduling optimization
- **Monthly Quality Metrics**: 1-hour refresh for compliance reporting
- **Patient Flow Analytics**: Automatic updates as appointments are scheduled/completed

In [ ]:
DROP SCHEMA IDENTIFIER($schema_name);